# INSEA 2026 data challenge: starter notebook

Will this used car sell within 30 days? This notebook loads the competition data, looks at one ad, trains the kit's
baseline and writes a submission you can send from the **Output** tab of a saved version.

To work from it: **Copy & Edit** (top right). For a GPU: **Settings > Accelerator**. To download a pretrained model:
**Settings > Internet: on**. The same code lives in the starter kit repository, where your team's final code must be.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd

COMPETITION = "multimodal-predictive-ml-data-challenge"
INPUT = Path("/kaggle/input") / COMPETITION
WORK = Path("/kaggle/working")
print(sorted(p.name for p in INPUT.iterdir()))

In [ ]:
# the photos: a folder if Kaggle unpacked images.zip, else we unpack it once into /kaggle/working
import zipfile
IMAGES = INPUT / "images"
if not IMAGES.exists():
    IMAGES = WORK / "images"
    if not IMAGES.exists():
        with zipfile.ZipFile(INPUT / "images.zip") as z:
            z.extractall(WORK)
print(IMAGES, len(list(IMAGES.glob("*.jpg"))), "photos")

In [ ]:
train = pd.read_csv(INPUT / "train.csv")
test = pd.read_csv(INPUT / "test.csv")
print(train.shape, test.shape, "train sale rate", round(train.sold_30d.mean(), 3))
train.head(3)

## One ad, three sources

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def photo_names(row):
    f = row["image_files"]
    return [] if pd.isna(f) or not f else f.split(";")

ad = train[train.image_files.notna()].iloc[0]
print(ad.drop(["title", "description", "image_files"]).to_string(), "\n")
print("TITLE:", ad.title, "\n\nDESCRIPTION:", ad.description)
names = photo_names(ad)
fig, axes = plt.subplots(1, max(len(names), 1), figsize=(3 * max(len(names), 1), 3))
for ax, name in zip(np.atleast_1d(axes), names):
    ax.imshow(Image.open(IMAGES / name)); ax.set_title(name, fontsize=8); ax.axis("off")
plt.show()

## Baseline: logistic regression on the table, complete rows only

Identical to `kit/train.py` in the repository. It ignores the text and the photos, drops every training ad with a missing
value, and gives the train sale rate to the test ads with a missing value. Each of these choices is a place to start.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = ["reg_year", "km", "engine_l", "price_mad", "seats", "doors", "post_month"]
CATEGORICAL = ["make", "model", "gearbox", "fuel", "body_type", "color", "city", "seller_type"]
cols = NUMERIC + CATEGORICAL
base_rate = train.sold_30d.mean()
complete = train[cols].notna().all(axis=1)
X, y = train.loc[complete, cols], train.loc[complete, "sold_30d"]
print(f"{complete.sum():,} complete training ads out of {len(train):,}")

model = Pipeline([
    ("prep", ColumnTransformer([("num", StandardScaler(), NUMERIC),
                                ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10), CATEGORICAL)])),
    ("clf", LogisticRegression(C=1.0, max_iter=2000)),
])
cv = -cross_val_score(model, X, y, cv=5, scoring="neg_log_loss")
print(f"5-fold log loss on the complete rows: {cv.mean():.4f} (+/- {cv.std():.4f})")
model.fit(X, y)

test_complete = test[cols].notna().all(axis=1)
p = np.full(len(test), base_rate)
p[test_complete.values] = model.predict_proba(test.loc[test_complete, cols])[:, 1]
p = np.clip(p, 0.01, 0.99)
print(f"{(~test_complete).sum():,} test ads with a missing value get the train sale rate")

## Write and check the submission

In [ ]:
sub = pd.DataFrame({"id": test.id, "sold_30d": p})
assert list(sub.columns) == ["id", "sold_30d"] and len(sub) == 19_919
assert set(sub.id) == set(test.id) and not sub.id.duplicated().any()
assert sub.sold_30d.notna().all() and sub.sold_30d.between(0, 1).all()
sub.to_csv(WORK / "submission.csv", index=False)
print(f"submission.csv: {len(sub):,} rows, mean {sub.sold_30d.mean():.3f}, min {sub.sold_30d.min():.3f}, max {sub.sold_30d.max():.3f}")

To submit: **Save Version** (top right, "Save & Run All"), then open the saved version, tab **Output**, and click
**Submit to competition** next to `submission.csv`. One submission per day per team.

## GPU check

In [ ]:
import torch
print("cuda available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU: Settings > Accelerator")